In [2]:
from pymongo import MongoClient


In [3]:

client = MongoClient(host='localhost', port=27017)

print(client.list_database_names())

['admin', 'beauty', 'config', 'local', 'mydb', 'school']


In [ ]:
db = client['beauty']
products = db['products']
reviews = db['reviews']
id_db = client['login']



In [23]:
import jsonlines
from tqdm import tqdm
with jsonlines.open('All_Beauty.jsonl') as f:
    for post in tqdm(f.iter()):
        reviews.insert_one(post)


701528it [02:50, 4117.58it/s]


In [24]:
import jsonlines
from tqdm import tqdm
with jsonlines.open('meta_All_Beauty.jsonl') as f:
    for post in tqdm(f.iter()):
        products.insert_one(post)


112590it [00:29, 3805.71it/s]


In [ ]:
review_5 = reviews.find({'rating': 5.0})
for rev in review_5[:5]:
    print(rev)

{'_id': ObjectId('670cc7bec16defd28c805dc6'), 'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': True}
{'_id': ObjectId('670cc7bec16defd28c805dc8'), 'rating': 5.0, 'title': 'Yes!', 'text': 'Smells good, feels great!', 'images': [], 'asin': 'B07PNNCSP9', 'parent_asin': 'B097R46CSY', 'user_id': 'AE74DYR3QUGVPZJ3P7RFWBGIX7XQ', 'timestamp': 1589665266052, 'helpful_vote': 2, 'verified_purchase': True}
{'_id': ObjectId('670cc7bec16defd28c805dca'), 'rating': 5.0, 'title': 'A+', 'text': 'Love it',

In [14]:
from pymongo import MongoClient
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix

# Connect to MongoDB
client = MongoClient(host='localhost', port=27017)
db = client["beauty"]  # Replace with your database name
collection = db["reviews"]  # Replace with your collection name
products = db["products"]  # Replace with your collection name

# Fetch review data
reviews = list(collection.find({}, {'user_id': 1, 'asin': 1, 'rating': 1}))
# Convert to DataFrame

df = pd.DataFrame(reviews)



In [15]:

# Convert user_id and asin to categorical data types to reduce memory usage
df['user_id'] = df['user_id'].astype('category')
df['asin'] = df['asin'].astype('category')

# Map categories to codes (numerical indices) for sparse matrix creation
df['user_id_code'] = df['user_id'].cat.codes
df['asin_code'] = df['asin'].cat.codes

# Create a sparse user-item matrix
user_item_sparse = csr_matrix((df['rating'], (df['user_id_code'], df['asin_code'])))

# Calculate item-to-item cosine similarity in a sparse format
item_similarity_sparse = cosine_similarity(user_item_sparse.T, dense_output=False)


In [31]:
from IPython.display import Image, display

def print_product(product_asin):
        
    reviews = list(products.find({'parent_asin': product_asin}))[0]
    url = reviews['images'][0]['large']
    print(reviews['title'])
    print(reviews['price'])
    display(Image(url=url))


def recommend_products(product_asin, num_recommendations=5):
    """
    Recommends products similar to the given product ASIN based on similarity scores.

    :param product_asin: ASIN of the product for which recommendations are to be made
    :param num_recommendations: Number of similar products to recommend
    :return: List of recommended products with similarity scores
    """
    # Fetch review data
    # Ensure the ASIN exists in the database
    if product_asin not in df['asin'].cat.categories:
        print("Product not found in the database.")
        return None

    # Get the product index code
    product_index = df['asin'].cat.categories.get_loc(product_asin)
    
    # Extract similarity scores for the specified product
    similar_indices = item_similarity_sparse[product_index].toarray().ravel().argsort()[::-1]
    similar_items = [(df['asin'].cat.categories[i], item_similarity_sparse[product_index, i]) 
                     for i in similar_indices[1:num_recommendations + 1]]

    return similar_items

# Example usage:
product_asin = 'B07WFSQXL5'  # Replace with the ASIN of a product you want recommendations for
recommendations = recommend_products(product_asin)
print("User products")
print_product(product_asin)
print("Recommended products:")
for item, score in recommendations:
    print_product(item)
    print()
    print(f"Similarity Score: {score}")

User products
Nail File, 12 Pcs Nail Files and Buffers 7 Steps Washable Emery Boards for Nails Professional Manicure Tools, Makeup Essential
None


Recommended products:
lixitur genmai SB
None



Similarity Score: 0.0
SWEETV Rhinestone Tiara Birthday Crown Princess Party Hat Hair Accessories 15/16/18/21/30/40/50/60/70th Birthday Gift, 60th
None



Similarity Score: 0.0
DatingDay 6-Way Nail File and Buffer Block (6 Pcs)
None



Similarity Score: 0.0
COKOHAPPY 8 Sheets Temporary Tattoo 27+ Different Designs for Men Women Arm Shoulder Flower, Eagle, Bear, Tiger, Owl, Wolf, Elephant, Deer
None



Similarity Score: 0.0
MANDI HOME Retro Court U-shaped Metal Hairpin Hair Clip Clamps Accessories 5 inches Fork Hairpin Barrettes Pin Ponytail Holder with Crystal Stones (Colorful)
None



Similarity Score: 0.0
